## Agent Implementation with Langchain

In [1]:
import os
from langchain_community.utilities import OpenWeatherMapAPIWrapper
os.environ["OPENWEATHERMAP_API_KEY"] = "6a8b0ac166a37e2b7a38e64416b3c3fe"
weather = OpenWeatherMapAPIWrapper()

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("azure_openai:gpt-4o-mini")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from langchain.agents import create_agent

agent = create_agent(model = model,
                     tools=[weather.run],
                     system_prompt="You are a helpful agent, choose the tool selection appropriately if needed.")

agent.invoke({"messages":[{"role":"user","content":"what is AI?"}]})

{'messages': [HumanMessage(content='what is AI?', additional_kwargs={}, response_metadata={}, id='3b303046-8286-447f-a7f0-e4638a6a135d'),
  AIMessage(content='Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn. These systems are designed to perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, solving problems, and making decisions. \n\nAI encompasses various subfields, including:\n\n1. **Machine Learning**: The use of algorithms and statistical models that enable computers to perform tasks without explicit instructions by learning from patterns and data.\n\n2. **Natural Language Processing (NLP)**: The ability of a computer to understand, interpret, and respond to human language in a way that is both valuable and meaningful.\n\n3. **Computer Vision**: The field that enables machines to interpret and make decisions based on visual data from the wo

In [4]:
agent.invoke({"messages":[{"role":"user","content":"what is weather today in mumbai?"}]})

{'messages': [HumanMessage(content='what is weather today in mumbai?', additional_kwargs={}, response_metadata={}, id='09316363-e92c-4243-b53a-ca23f5423c34'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 68, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh4XgPRwGmRGFo63bbFeiXEc2aoi', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'sever

### Multi tool Agent

In [5]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [6]:
agent = create_agent(model = model,
                     tools=[weather.run,wikipedia],
                     system_prompt="You are a helpful agent, choose the tool selection appropriately if needed.")

agent.invoke({"messages":[{"role":"user","content":"what is AI?"}]})

{'messages': [HumanMessage(content='what is AI?', additional_kwargs={}, response_metadata={}, id='e28051b2-866b-4238-8ae0-d037122e2106'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 122, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh4acoeZsfGAfWxTuVwK5WkKmLUv', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], '

In [7]:
agent.invoke({"messages":[{"role":"user","content":"findout the capital city of Brunei and the weather information there."}]})

{'messages': [HumanMessage(content='findout the capital city of Brunei and the weather information there.', additional_kwargs={}, response_metadata={}, id='17af3e7b-7b59-48ac-9202-1b11b4777f23'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 132, 'total_tokens': 177, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh4meTRgZrcCxu5UdWpuWHUMh7hX', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'},

### Enhancing Agent capabilities
- Adding memory checkpointer
- adding response format for output
- Adding a middleware for Human in the Loop

In [8]:
# adding a memory checkpointer
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

from langchain.agents import create_agent

agent = create_agent(model = model,
                     tools=[weather.run],
                     checkpointer=checkpointer,
                     system_prompt="You are a helpful agent, choose the tool selection appropriately if needed.")

In [9]:
config = {"configurable":{"thread_id":"123"}}

agent.invoke({"messages":[{"role":"user","content":"I live in Pune; "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='a305cedd-5633-4c65-85f7-7e8666c471a6'),
  AIMessage(content='How can I assist you today with information related to Pune? Are you looking for weather updates, local news, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 66, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh8XY09yQpniXPhDg0h6gGggsq0Q', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False

In [10]:
agent.invoke({"messages":[{"role":"user","content":"Help me with history of my city "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='a305cedd-5633-4c65-85f7-7e8666c471a6'),
  AIMessage(content='How can I assist you today with information related to Pune? Are you looking for weather updates, local news, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 66, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh8XY09yQpniXPhDg0h6gGggsq0Q', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False

In [11]:
agent.invoke({"messages":[{"role":"user","content":"what is the weather there today? "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='a305cedd-5633-4c65-85f7-7e8666c471a6'),
  AIMessage(content='How can I assist you today with information related to Pune? Are you looking for weather updates, local news, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 66, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh8XY09yQpniXPhDg0h6gGggsq0Q', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False

In [12]:
# Response format enforcement
from dataclasses import dataclass
@dataclass
class agentresponse:
    weather_conditions:str |None=None

In [13]:
agent = create_agent(model = model,
                     tools=[weather.run],
                     checkpointer=checkpointer,
                     response_format=agentresponse,
                     system_prompt="You are a helpful agent, choose the tool selection appropriately if needed.")

In [15]:
output =  agent.invoke({"messages":[{"role":"user","content":"what is the weather there today? "}]},
             config=config)

In [19]:
output['structured_response'].weather_conditions

'The current weather in Pune is overcast with a temperature of 30.82°C (feels like 28.92°C). The humidity is at 20%, and there is a wind speed of 2.32 m/s.'

In [17]:
output

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='a305cedd-5633-4c65-85f7-7e8666c471a6'),
  AIMessage(content='How can I assist you today with information related to Pune? Are you looking for weather updates, local news, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 66, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DCh8XY09yQpniXPhDg0h6gGggsq0Q', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False

In [ ]:
# Middleware for HITL

from langchain.agents.middleware import HumanInTheLoopMiddleware

mw1 = HumanInTheLoopMiddleware(interrupt_on={
    "weather":False,
    "wikipedia":True
},
description_prefix="Tool execution pending approval")

In [32]:
agent = create_agent(model = model,
                     tools=[weather.run,wikipedia],
                     checkpointer=checkpointer,
                     middleware=[mw1],
                     system_prompt="You are a helpful agent, choose the tool selection appropriately if needed.")



In [33]:
config = {"configurable":{"thread_id":"12345"}}

agent.invoke({"messages":[{"role":"user","content":"I live in Pune; "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='0491f0c4-59f1-44e4-857e-aa5baa55d561'),
  AIMessage(content='How can I assist you today regarding Pune? Would you like to know about the weather, places to visit, or anything specific?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 124, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DChTkWGxYnXTXjHgn8Nk5EUYMtCdT', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, '

In [34]:
agent.invoke({"messages":[{"role":"user","content":"Tell me more about my city "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='0491f0c4-59f1-44e4-857e-aa5baa55d561'),
  AIMessage(content='How can I assist you today regarding Pune? Would you like to know about the weather, places to visit, or anything specific?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 124, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DChTkWGxYnXTXjHgn8Nk5EUYMtCdT', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, '

In [35]:
from langgraph.types import Command
agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='0491f0c4-59f1-44e4-857e-aa5baa55d561'),
  AIMessage(content='How can I assist you today regarding Pune? Would you like to know about the weather, places to visit, or anything specific?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 124, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DChTkWGxYnXTXjHgn8Nk5EUYMtCdT', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, '

In [36]:
agent.invoke({"messages":[{"role":"user","content":"whats the weather in my city? "}]},
             config=config)

{'messages': [HumanMessage(content='I live in Pune; ', additional_kwargs={}, response_metadata={}, id='0491f0c4-59f1-44e4-857e-aa5baa55d561'),
  AIMessage(content='How can I assist you today regarding Pune? Would you like to know about the weather, places to visit, or anything specific?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 124, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DChTkWGxYnXTXjHgn8Nk5EUYMtCdT', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, '